# Day 05：JSON 解析与大模型 API 数据交换 —— 智枢平台 JSON 引擎实战

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 5 天！我是你的主讲兼 Tech Lead 张工。很多同学一听“调用大模型 API”就觉得很底层、很难，其实所有大模型接口（OpenAI、DeepSeek、Qwen）和我们 Python 程序之间交互的唯一通用语言就是 **JSON**。今天我们不讲枯燥的格式定义，直接带大家手写智枢平台的 `json_parser.py`：解析真实风格的 API 响应、用 `ensure_ascii=False` 保存中文多轮对话、并遍历嵌套字典汇总 Token 消耗账单！

## 1. 第一步：认识标准大模型 API 响应与 `json.loads()`
当你向 DeepSeek 或 OpenAI 发送一条消息后，服务器返回的是一段**纯文本格式的 JSON 字符串**。
在 Python 中，我们使用标准库 `json.loads()`（load string）把这段字符串反序列化为 Python 字典 `dict`，这样就能像查字典一样提取模型回复和 Token 计费信息。

In [1]:
import json

# 模拟一段真实的 OpenAI / DeepSeek Chat Completions API 返回的 JSON 字符串
raw_api_json = '''
{
  "id": "chatcmpl-day05-888",
  "object": "chat.completion",
  "created": 1731456000,
  "model": "deepseek-chat",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "你好！我是智枢平台 AI 助手，已准备好为你解析 JSON 数据。"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 28,
    "completion_tokens": 22,
    "total_tokens": 50
  }
}
'''

# 1. 使用 json.loads() 将 JSON 字符串解析为 Python 字典
data = json.loads(raw_api_json)
print('解析后的根数据类型:', type(data))

# 2. 逐层提取嵌套字段：choices 是列表，第 0 项里包含 message 字典
reply_content = data['choices'][0]['message']['content']
total_tokens = data['usage']['total_tokens']

print('模型名称  :', data['model'])
print('模型回复  :', reply_content)
print('总 Token  :', total_tokens)

解析后的根数据类型: <class 'dict'>
模型名称  : deepseek-chat
模型回复  : 你好！我是智枢平台 AI 助手，已准备好为你解析 JSON 数据。
总 Token  : 50


## 2. 第二步：封装健壮的响应提取函数 `extract_llm_response`
在企业级工程中，我们不能每次都手写 `data['choices'][0]['message']['content']`，万一返回格式异常或缺少字段，程序就会直接 `KeyError` 崩溃。
下面我们把解析逻辑封装成 `extract_llm_response(raw_json)`，将深层嵌套结构拍平为干净的字典。

In [2]:
def extract_llm_response(raw_json: str) -> dict:
    """解析大模型 API 响应 JSON 字符串，提取核心业务字段。"""
    # 第一步：尝试反序列化 JSON，若语法非法则抛出清晰的 ValueError
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        raise ValueError(f"LLM 响应 JSON 解析失败: {exc}") from exc

    # 第二步：校验 choices 列表是否存在且非空
    choices = data.get('choices')
    if not isinstance(choices, list) or len(choices) == 0:
        raise ValueError("LLM 响应缺少有效的 choices 列表")

    # 第三步：安全读取嵌套的 message.content 与 usage 统计
    content = choices[0].get('message', {}).get('content', '')
    usage = data.get('usage', {})

    # 第四步：返回拍平后的标准字典
    return {
        'id': str(data.get('id', '')),
        'model': str(data.get('model', '')),
        'content': str(content),
        'prompt_tokens': int(usage.get('prompt_tokens', 0)),
        'completion_tokens': int(usage.get('completion_tokens', 0)),
        'total_tokens': int(usage.get('total_tokens', 0)),
    }

# 调用函数验证效果
parsed = extract_llm_response(raw_api_json)
for k, v in parsed.items():
    print(f"{k:18s}: {v}")

id                : chatcmpl-day05-888
model             : deepseek-chat
content           : 你好！我是智枢平台 AI 助手，已准备好为你解析 JSON 数据。
prompt_tokens     : 28
completion_tokens : 22
total_tokens      : 50


## 3. 第三步：中文会话持久化（`json.dump` 与 `ensure_ascii=False`）
智能体需要记住多轮对话历史（`messages: list[dict]`）。把对话保存到本地文件时，新手最常踩的坑是：
- 忘记加 `ensure_ascii=False`，导致中文全部变成 `\u4f60\u597d`，人工排查日志时根本看不懂；
- 忘记加 `indent=2`，导致整个 JSON 挤成一行。
下面我们用 `tempfile` 演示标准的保存与加载流程。

In [3]:
import tempfile
from pathlib import Path

def save_session_json(filepath: Path, messages: list[dict]) -> None:
    """将多轮对话历史保存为格式化、保留中文的 JSON 文件。"""
    target = Path(filepath)
    # 自动创建不存在的父级目录
    target.parent.mkdir(parents=True, exist_ok=True)
    # 使用 utf-8 编码，并设置 ensure_ascii=False 保留中文原样
    with target.open('w', encoding='utf-8') as f:
        json.dump(messages, f, ensure_ascii=False, indent=2)

def load_session_json(filepath: Path) -> list[dict]:
    """从本地 JSON 文件恢复多轮对话历史。"""
    target = Path(filepath)
    with target.open('r', encoding='utf-8') as f:
        return json.load(f)

# 构造一段包含中文的多轮对话
chat_messages = [
    {'role': 'system', 'content': '你是星辰数科智枢平台企业助手。'},
    {'role': 'user', 'content': '请帮我总结今天的 JSON 课程重点。'},
    {'role': 'assistant', 'content': parsed['content']},
]

# 在临时目录中演示保存、查看原始文本、重新加载
with tempfile.TemporaryDirectory() as tmp_dir:
    session_path = Path(tmp_dir) / 'session_day05.json'
    save_session_json(session_path, chat_messages)

    print('=== 磁盘上的 JSON 原文（中文清晰可读）===')
    print(session_path.read_text(encoding='utf-8'))

    restored = load_session_json(session_path)
    print('=== 重新加载后的数据条数 ===', len(restored))

=== 磁盘上的 JSON 原文（中文清晰可读）===
[
  {
    "role": "system",
    "content": "你是星辰数科智枢平台企业助手。"
  },
  {
    "role": "user",
    "content": "请帮我总结今天的 JSON 课程重点。"
  },
  {
    "role": "assistant",
    "content": "你好！我是智枢平台 AI 助手，已准备好为你解析 JSON 数据。"
  }
]
=== 重新加载后的数据条数 === 3


## 4. 第四步：嵌套字典遍历 —— Token 用量聚合器
在企业生产环境中，一次复杂任务（比如 Agent 思考 + 调工具 + 总结）往往会连续调用多次大模型 API。
我们需要遍历每一次响应字典，钻取嵌套的 `usage` 字段，汇总出总请求数与各类型 Token 消耗。

In [4]:
def summarize_usage(responses: list[dict]) -> dict[str, int]:
    """遍历多次大模型响应字典，汇总 Token 消耗总量。"""
    total_prompt = 0
    total_completion = 0
    total_all = 0

    for resp in responses:
        # 安全获取嵌套的 usage 字典
        usage = resp.get('usage', {})
        prompt = int(usage.get('prompt_tokens', 0))
        completion = int(usage.get('completion_tokens', 0))
        total = int(usage.get('total_tokens', prompt + completion))

        # 累加各维度 Token 数
        total_prompt += prompt
        total_completion += completion
        total_all += total

    return {
        'request_count': len(responses),
        'prompt_tokens': total_prompt,
        'completion_tokens': total_completion,
        'total_tokens': total_all,
    }

# 模拟 3 次连续 API 调用的原始响应
api_responses = [
    json.loads(raw_api_json),
    {'id': 'chatcmpl-002', 'usage': {'prompt_tokens': 120, 'completion_tokens': 45, 'total_tokens': 165}},
    {'id': 'chatcmpl-003', 'usage': {'prompt_tokens': 80, 'completion_tokens': 30, 'total_tokens': 110}},
]

summary = summarize_usage(api_responses)
print('今日智枢平台 Token 消耗汇总:', summary)

今日智枢平台 Token 消耗汇总: {'request_count': 3, 'prompt_tokens': 228, 'completion_tokens': 97, 'total_tokens': 325}
